# Online Shopping Behavior Analysis - Reproducible Notebook

This notebook walks through the key steps of the online shopping behavior analysis project. It covers data loading, exploratory data analysis (EDA), feature engineering, model training, and interpretation. By following this notebook, you can reproduce the analysis and adapt it for further experimentation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve

# Set visualization styles
sns.set(style='whitegrid')


## Data Loading

Load the `online_shoppers_intention.csv` dataset. This dataset contains session-level information about online shopping behavior and whether a session resulted in a purchase. Adjust the file path if necessary.


In [ ]:
# Load the dataset
file_path = 'data/online_shoppers_intention.csv'
df = pd.read_csv(file_path)

# Display basic information about the dataset
print('Dataset shape:', df.shape)
display(df.head())


## Exploratory Data Analysis (EDA)

We examine distributions of key numeric variables, investigate relationships between features and the target variable (`Revenue`), and visualize correlations. Replace or extend these examples as needed.


In [ ]:
# Summary statistics
numeric_cols = df.select_dtypes(include=['int64', 'float64'])
df[numeric_cols.columns].describe()

# Plot histograms for a few important features
features_to_plot = ['PageValues', 'BounceRates', 'ExitRates', 'ProductRelated_Duration']
plt.figure(figsize=(12, 8))
for i, col in enumerate(features_to_plot, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

# Correlation matrix
corr = numeric_cols.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


## Feature Engineering

Create new features to capture combined effects. For example:
* `TotalDuration` – sum of administrative, informational, and product-related durations.
* `TotalPages` – total number of pages visited.
* `AvgDurationPerPage` – average duration per page (handling division by zero).
* Cyclic encoding for the `Month` variable.

These engineered features often improve model performance.


In [ ]:
# Create total and average duration features
df['TotalDuration'] = df['Administrative_Duration'] + df['Informational_Duration'] + df['ProductRelated_Duration']
df['TotalPages'] = df['Administrative'] + df['Informational'] + df['ProductRelated']
df['AvgDurationPerPage'] = df['TotalDuration'] / df['TotalPages']
df['AvgDurationPerPage'] = df['AvgDurationPerPage'].fillna(0)

# Cyclic encoding for month
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df['Month_Num'] = df['Month'].map(lambda x: month_order.index(x))
df['Month_Sin'] = np.sin(2 * np.pi * df['Month_Num'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month_Num'] / 12)

# Drop original month column
prepared_df = df.drop(columns=['Month'])


## Train–Test Split and Scaling

Separate the features and target, split into training and testing sets, and scale numerical features. Apply SMOTE to balance classes in the training data.


In [ ]:
# Separate features and target
target_col = 'Revenue'
X = prepared_df.drop(columns=[target_col])
y = prepared_df[target_col]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE for class balancing
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)


## Model Training

Use a Random Forest classifier as a baseline model. Other models such as logistic regression, gradient boosting, or LightGBM can also be tried. Train the model on the balanced training data.


In [ ]:
# Train a Random Forest classifier
clf = RandomForestClassifier(random_state=42, n_estimators=200)
clf.fit(X_train_balanced, y_train_balanced)


## Evaluation

Evaluate the model using accuracy, precision, recall, F1-score, ROC curve, and precision-recall curve. Visualize the ROC and PR curves and print the classification report and confusion matrix.


In [ ]:
# Predictions and probabilities
y_pred = clf.predict(X_test_scaled)
y_proba = clf.predict_proba(X_test_scaled)[:, 1]

# Classification report and confusion matrix
print('Classification Report:
', classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

# Precision-Recall curve
precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall, precision)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision, label=f'AUC = {pr_auc:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()


## Feature Importance

Examine which features have the greatest impact on the model’s predictions.


In [ ]:
# Create a feature importance DataFrame
importances = clf.feature_importances_
feature_names = X.columns
feature_importance = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance = feature_importance.sort_values('importance', ascending=False)

# Plot the top 15 features
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
plt.title('Top 15 Feature Importances')
plt.show()

feature_importance.head(20)


## Interpretation and Next Steps

The plots and metrics above reveal which factors influence purchase conversion the most. You can compare different algorithms (e.g., logistic regression, XGBoost) and perform hyperparameter tuning to improve performance. Additionally, clustering techniques (such as K-Means) can be applied for customer segmentation, and sentiment analysis of customer reviews can provide qualitative insights to complement the quantitative analysis.
